# 1. What you'll learn

        - hold compute budget and CV folds constant
- visualize best-so-far validation score
- show search overfitting when the validation set is reused too aggressively

        **The one question this notebook answers:** Under an identical 24-candidate budget, how differently do grid, random, and Bayesian search explore an SVM?

# 2. Setup

This lesson keeps grid, random, and optuna search small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import time
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score,GridSearchCV,RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.stats import loguniform

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The breast-cancer dataset is split once into development and untouched test. Every search evaluates exactly 24 RBF-SVM configurations with the same three folds.

**Small example:** With a budget of five trials, a 5×5 grid cannot cover 25 combinations. Random search samples five combinations; Bayesian search uses earlier scores to choose later ones.

In [ ]:
b=load_breast_cancer();X=b.data;y=(b.target==0).astype(int);X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED);print("Development/test:",X_dev.shape,X_test.shape,"malignant share:",round(y.mean(),3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(["development","test"],[len(X_dev),len(X_test)],color=["#176b66","#d38b45"]);axes[0].set_title("Locked test")
axes[1].hist(np.std(X_dev,0),bins=20,color="#176b66");axes[1].set_title("Feature scales require pipeline")
axes[2].imshow(np.log10(np.outer(np.logspace(-2,3,6),np.logspace(-4,1,4))),cmap="mako");axes[2].set_title("Grid covers fixed lattice");plt.tight_layout();plt.show()

**Takeaway:** The test set is excluded from all 72 candidate evaluations.

**Takeaway:** Feature scales span orders of magnitude, so scaling belongs inside CV.

**Takeaway:** A grid spends trials on every Cartesian crossing whether promising or not.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
cv=StratifiedKFold(3,shuffle=True,random_state=SEED);pipe=make_pipeline(StandardScaler(),SVC(kernel="rbf",probability=True,random_state=SEED));budget=24;print("Shared folds:",cv.get_n_splits(),"candidates/method:",budget)

# 6. Train

        Training turns the assumptions behind grid, random, and optuna search into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Define one search space and one candidate budget.
2. Score candidates on identical stratified folds.
3. Grid covers a fixed lattice; random samples independently; Optuna adapts using earlier trials.
4. Refit each method's winner on all development rows and evaluate test once.

In [ ]:
t=time.perf_counter();grid=GridSearchCV(pipe,{"svc__C":np.logspace(-2,3,6),"svc__gamma":np.logspace(-4,1,4)},scoring="roc_auc",cv=cv,n_jobs=1).fit(X_dev,y_dev);grid_time=time.perf_counter()-t
t=time.perf_counter();random_search=RandomizedSearchCV(pipe,{"svc__C":loguniform(1e-2,1e3),"svc__gamma":loguniform(1e-4,10)},n_iter=budget,scoring="roc_auc",cv=cv,random_state=SEED,n_jobs=1).fit(X_dev,y_dev);random_time=time.perf_counter()-t
optuna.logging.set_verbosity(optuna.logging.WARNING);trail=[]
def objective(trial):
 c=trial.suggest_float("C",1e-2,1e3,log=True);g=trial.suggest_float("gamma",1e-4,10,log=True);score=cross_val_score(make_pipeline(StandardScaler(),SVC(C=c,gamma=g,probability=True,random_state=SEED)),X_dev,y_dev,scoring="roc_auc",cv=cv,n_jobs=1).mean();trail.append(score);return score
t=time.perf_counter();study=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=SEED));study.optimize(objective,n_trials=budget);optuna_time=time.perf_counter()-t;opt_model=make_pipeline(StandardScaler(),SVC(C=study.best_params["C"],gamma=study.best_params["gamma"],probability=True,random_state=SEED)).fit(X_dev,y_dev);print("Best CV:",round(grid.best_score_,3),round(random_search.best_score_,3),round(study.best_value,3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
methods={"grid":grid.best_estimator_,"random":random_search.best_estimator_,"Optuna":opt_model};test_auc={n:roc_auc_score(y_test,m.predict_proba(X_test)[:,1]) for n,m in methods.items()};grid_curve=np.maximum.accumulate(grid.cv_results_["mean_test_score"]);random_curve=np.maximum.accumulate(random_search.cv_results_["mean_test_score"]);opt_curve=np.maximum.accumulate(trail)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(grid_curve,label="grid");axes[0].plot(random_curve,label="random");axes[0].plot(opt_curve,label="Optuna");axes[0].set(title="Best CV score so far",xlabel="candidate");axes[0].legend()
axes[1].bar(test_auc.keys(),test_auc.values(),color=["#176b66","#d38b45","#768f8b"]);axes[1].axhline(.5,ls="--",color="black");axes[1].set_ylim(.45,1);axes[1].set_title("One untouched test")
axes[2].bar(["grid","random","Optuna"],[grid_time,random_time,optuna_time],color=["#176b66","#d38b45","#768f8b"]);axes[2].set(title="Wall time",ylabel="seconds");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Test AUC:",{k:round(v,3) for k,v in test_auc.items()})

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Repeatedly maximizing the same noisy CV estimate overfits model selection; a larger search is not automatically more trustworthy. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);noise_scores=[]
for j in range(250):
 fake=rng.normal(size=len(y_dev));noise_scores.append(roc_auc_score(y_dev,fake))
print(f"One random score expectation≈.5; best of 250 reused-validation random scores={max(noise_scores):.3f}. More search can select noise even when every candidate has zero signal.")

**Use this when…** hyperparameters materially affect generalization and you can reserve nested or untouched evaluation data.

        **Don't use this when…** the dataset is too small for reliable folds, compute budgets differ, or the final test is consulted during search.

        ## Try this

        1. Repeat all searches with a second CV seed.
2. Use linear rather than log-uniform sampling and inspect wasted trials.
3. Nest the entire search inside outer CV.